# ĐỒ ÁN BIG DATA: HỆ KHUYẾN NGHỊ SẢN PHẨM (RETAILROCKET DATASET)
## NOTEBOOK 01: PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA) & TIỀN XỬ LÝ
**Thành viên thực hiện:** PHƯƠNG (Chuyên viên Phân tích & Kỹ sư Dữ liệu)

### Nội dung thực hiện:
1. Nạp và khảo sát cấu trúc tập dữ liệu `events.csv`.
2. Phân tích phân bố hành vi (View, AddToCart, Transaction) & Phễu chuyển đổi (Conversion Funnel).
3. Phân tích mật độ tương tác theo chu kỳ thời gian (Giờ & Thứ trong tuần).
4. Đánh giá tính chất Long-tail của người dùng và sản phẩm.
5. Xác định Top 10 sản phẩm thịnh hành (Cold-Start candidates).
6. Xây dựng logic quy đổi phản hồi ẩn (Implicit Feedback) thành điểm số.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập cấu hình trực quan hóa
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'font.sans-serif': 'Segoe UI', 'font.size': 11})

### 1. Nạp dữ liệu sự kiện tương tác RetailRocket

In [ ]:
data_path = '../data/raw/events.csv'
df = pd.read_csv(data_path)
print(f'Tổng số dòng dữ liệu: {len(df):,}')
df.head()

### 2. Tiền xử lý trường thời gian và thống kê cơ bản

In [ ]:
df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.day_name()

print('Số người dùng unique:', df['visitorid'].nunique())
print('Số sản phẩm unique:', df['itemid'].nunique())
print('Khoảng thời gian:', df['datetime'].min(), '->', df['datetime'].max())

### 3. Phân bố các loại hành vi người dùng (Event Distribution)

In [ ]:
event_counts = df['event'].value_counts()
display(event_counts.to_frame('Số lượng'))

plt.figure(figsize=(7, 4))
sns.barplot(x=event_counts.index, y=event_counts.values, palette='Blues_r', edgecolor='black')
plt.yscale('log')
plt.title('Phân bố loại hành vi người dùng (Thang Logarit)', fontweight='bold')
plt.ylabel('Số lượng (Log scale)')
plt.show()

### 4. Phân tích tương tác theo khung giờ trong ngày

In [ ]:
hourly = df.groupby(['hour', 'event']).size().unstack(fill_value=0)
plt.figure(figsize=(10, 4.5))
plt.plot(hourly.index, hourly['view'], label='View', marker='o', color='#3B82F6')
plt.title('Mật độ tương tác theo từng giờ trong ngày (24 Giờ)', fontweight='bold')
plt.xlabel('Khung giờ (UTC 0h - 23h)')
plt.ylabel('Số lượt xem')
plt.xticks(range(24))
plt.legend()
plt.show()

### 5. Hiện tượng Long-Tail & Bài toán Cold-Start

In [ ]:
user_interactions = df.groupby('visitorid').size()
single_int_pct = (user_interactions == 1).mean() * 100
print(f'Tỷ lệ người dùng chỉ tương tác 1 lần (Cold-Start): {single_int_pct:.2f}%')

# Top 10 sản phẩm thịnh hành nhất
top_items = df['itemid'].value_counts().head(10)
display(top_items.to_frame('Số lượt tương tác'))

### 6. Logic quy đổi điểm tương tác (Implicit Feedback Weighting)
Quy tắc gán trọng số: `view` = 1, `addtocart` = 3, `transaction` = 5.

In [ ]:
weights = {'view': 1, 'addtocart': 3, 'transaction': 5}
df['weight'] = df['event'].map(weights)
user_item_matrix = df.groupby(['visitorid', 'itemid'])['weight'].sum().reset_index()
user_item_matrix.columns = ['userId', 'itemId', 'rating']
print(f'Số cặp User-Item độc nhất: {len(user_item_matrix):,}')
user_item_matrix.head(10)